# Local Inference with Ollama

Needs `ollama serve` running.

- `list_models()` - what's installed
- `query_model(prompt, model)` - prompt in, reply out
- `unload(model)` - free it from memory

In [ ]:
import requests

BASE = "http://localhost:11434"

def list_models():
    """Return the names of all locally installed models."""
    models = requests.get(f"{BASE}/api/tags").json()["models"]
    return [m["name"] for m in models]

list_models()

In [ ]:
def query_model(prompt, model, think=None):
    """Send a prompt to any local model and return its reply.

    think=False hides reasoning on reasoning models (e.g. qwen3);
    leave it as None for models that don't support thinking.
    """
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }
    if think is not None:
        payload["think"] = think
    r = requests.post(f"{BASE}/api/chat", json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["message"]["content"]

In [ ]:
def unload(model):
    """Free a model from memory (keep_alive=0 unloads it immediately)."""
    requests.post(f"{BASE}/api/generate", json={"model": model, "keep_alive": 0})

In [ ]:
import time

model = "qwen3.8:27b-mlx"  # pick any name from list_models()
start = time.perf_counter()
reply = query_model("What do llamas eat?", model, think=False)
print(reply)
print(f"\n[{model}: {time.perf_counter() - start:.1f}s]")